In [1]:
from huggingface_hub import notebook_login
from datasets import load_dataset
import torch
from transformers import AutoTokenizer

notebook_login()

In [2]:
eli5 = load_dataset("dany0407/eli5_category", split="train[:5000]")
eli5 = eli5.train_test_split(test_size=0.2)
eli5["train"][0]

{'q_id': '7di2hz',
 'title': 'Why didnt Germans just go around the Berlin wall?',
 'selftext': '',
 'category': 'Other',
 'subreddit': 'explainlikeimfive',
 'answers': {'a_id': ['dpxz1e3', 'dpxyz7s', 'dpy3wyz', 'dpxyxws', 'dpxz00w'],
  'text': ["it completely surrounded west berlin. a full circle, as berlin was otherwise fully located in east Germany, so you couldn't go into west berlin to then emigrate to west Germany and other western states.",
   'The Berlin wall went all the way around the city of Berlin. It was meant to keep people from going from East Germany (which was controlled by the Soviets ...Russia...and surrounded the city of Berlin.) to west Berlin where they could get out to the free world.',
   "In addition to the responses you've already received, [here's a map]( URL_0 ). The famous concrete wall is the red bit that divides the two halves of the city; the grey sections show where the border installations cut off West Berlin from its hinterland, and instead of a iconic

In [3]:
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilgpt2")

In [4]:
eli5 = eli5.flatten()
eli5["train"][0]

{'q_id': '7di2hz',
 'title': 'Why didnt Germans just go around the Berlin wall?',
 'selftext': '',
 'category': 'Other',
 'subreddit': 'explainlikeimfive',
 'answers.a_id': ['dpxz1e3', 'dpxyz7s', 'dpy3wyz', 'dpxyxws', 'dpxz00w'],
 'answers.text': ["it completely surrounded west berlin. a full circle, as berlin was otherwise fully located in east Germany, so you couldn't go into west berlin to then emigrate to west Germany and other western states.",
  'The Berlin wall went all the way around the city of Berlin. It was meant to keep people from going from East Germany (which was controlled by the Soviets ...Russia...and surrounded the city of Berlin.) to west Berlin where they could get out to the free world.',
  "In addition to the responses you've already received, [here's a map]( URL_0 ). The famous concrete wall is the red bit that divides the two halves of the city; the grey sections show where the border installations cut off West Berlin from its hinterland, and instead of a iconi

In [5]:
def preprocess_function(examples):
    return tokenizer([" ".join(x) for x in examples["answers.text"]])

In [6]:
tokenized_eli5 = eli5.map(
    preprocess_function,
    batched=True,
    num_proc=4,
    remove_columns=eli5["train"].column_names,
)

Map (num_proc=4):   0%|          | 0/4000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1118 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1994 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (8074 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1618 > 1024). Running this sequence through the model will result in indexing errors


Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1031 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (2110 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (2727 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1091 > 1024). Running this sequence through the model will result in indexing errors


In [7]:
block_size = 128

def group_texts(examples):
    # Concatenate all texts.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, we could add padding if the model supported it 
    # instead of this drop, you can customize this part to your needs.
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    # Split by chunks of block_size.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

In [8]:
lm_dataset = tokenized_eli5.map(group_texts, batched=True, num_proc=4)

Map (num_proc=4):   0%|          | 0/4000 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

In [9]:
from transformers import DataCollatorForLanguageModeling

tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [10]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

model = AutoModelForCausalLM.from_pretrained("distilbert/distilgpt2")

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [11]:
training_args = TrainingArguments(
    output_dir="my_awesome_eli5_clm-model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    push_to_hub=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset["train"],
    eval_dataset=lm_dataset["test"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

trainer.train()

HfHubHTTPError: (Request ID: Root=1-69c6dfac-651fe8514a23b8c5435888f4;a4330b56-ee96-4c18-947c-0b72c350abf0)

403 Forbidden: You don't have the rights to create a model under the namespace "xXDarkLordXx".
Cannot access content at: https://huggingface.co/api/repos/create.
Make sure your token has the correct permissions.

In [ ]:
import math

eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

In [ ]:
trainer.push_to_hub()